# Opdracht 2 · Lineair model & diagnostiek
### Regressie-training · Antwoordmodel-versie

In deze opdracht bouw je je eerste echte model en leer je het te beoordelen. Je gaat:

1. Een **lineair regressiemodel** trainen op de data uit opdracht 1
2. De **coëfficiënten** bekijken en de twee sterkste effecten benoemen
3. Een **residuenplot** maken en lineariteit + homoscedasticiteit beoordelen
4. De **VIF** berekenen om multicollineariteit op te sporen
5. (Bonus) Vergelijken met een **Ridge**-model

> **Hoe werkt dit notebook?**
> De meeste cellen zijn al ingevuld. Op de plekken met **`# TODO`** vul jij iets in.
> We gebruiken dezelfde Diabetes-dataset als in opdracht 1, zodat je de baseline kunt verslaan.


## 0. Voorbereiding
Imports en data laden. Deze cellen hoef je niet aan te passen — gewoon uitvoeren.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams["figure.figsize"] = (7, 4)
print("Bibliotheken geladen.")

In [ ]:
# Data laden en splitsen (zelfde opzet als opdracht 1)
data = load_diabetes(as_frame=True)
df = data.frame
y = df["target"]
X = df.drop(columns=["target"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape[0], "rijen | Test:", X_test.shape[0], "rijen")
print("Features:", list(X.columns))

## 1. Een lineair model trainen
Een lineair regressiemodel zoekt de best passende rechte (eigenlijk: een vlak in meerdere dimensies)
door de coëfficiënten te kiezen die de fouten minimaliseren.

### TODO 1 — Train het model
Maak een `LinearRegression`-model, train het op de **train-set**, en laat het de **test-set** voorspellen.

> Tip: `model = LinearRegression()`, daarna `model.fit(X_train, y_train)`,
> en voorspellen met `model.predict(X_test)`.

In [ ]:
# ANTWOORD
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.3f}")

Vergelijk de **R²** met de baseline uit opdracht 1 (die was ongeveer **0**).
Het lineaire model zou duidelijk beter moeten presteren.

## 2. Coëfficiënten interpreteren
Elke feature krijgt een coëfficiënt: het effect op de target wanneer die feature met één eenheid toeneemt
(en de rest gelijk blijft). De features zijn hier vooraf geschaald, dus je mag de coëfficiënten
onderling vergelijken qua grootte.

### TODO 2 — Zet de coëfficiënten in een tabel
Maak een `pandas.Series` met de coëfficiënten (`model.coef_`) en de featurenamen (`X.columns`),
en sorteer op **absolute** grootte zodat de sterkste effecten bovenaan staan.

> Tip: `pd.Series(model.coef_, index=X.columns)` en daarna
> `.sort_values(key=abs, ascending=False)`.

In [ ]:
# ANTWOORD
coefs = pd.Series(model.coef_, index=X.columns)
coefs = coefs.sort_values(key=abs, ascending=False)
print(coefs)

Bekijk de twee coëfficiënten met de grootste absolute waarde. Een **positieve** coëfficiënt
betekent dat de feature de target verhoogt, een **negatieve** dat hij hem verlaagt.

> Noteer voor jezelf: welke twee features hebben het sterkste effect, en in welke richting?

## 3. Residuenplot
Een **residu** is het verschil tussen de werkelijke en de voorspelde waarde.
Door de residuen tegen de voorspelde waarden te plotten, controleer je twee aannames:
- **Lineariteit**: de punten zweven willekeurig rond nul, zonder kromme of patroon.
- **Homoscedasticiteit**: de spreiding is overal ongeveer gelijk (geen waaiervorm).

### TODO 3 — Bereken de residuen
Bereken de residuen op de test-set: werkelijke waarde min voorspelde waarde.

> Tip: `residuen = y_test - y_pred`.

In [ ]:
# ANTWOORD
residuen = y_test - y_pred
print("Eerste paar residuen:")
print(residuen.head())

De plot zelf is al voor je gemaakt — voer de cel uit en interpreteer hem.

In [ ]:
# Residuenplot (al ingevuld)
plt.scatter(y_pred, residuen, color="#4473C5", alpha=0.7, edgecolor="white")
plt.axhline(0, color="#D97733", linewidth=2)
plt.xlabel("Voorspelde waarde")
plt.ylabel("Residu (werkelijk - voorspeld)")
plt.title("Residuenplot")
plt.show()

> **Beoordeel de plot:**
> - Zweven de punten redelijk willekeurig rond de oranje nullijn? (→ lineariteit oké)
> - Is de spreiding overal ongeveer gelijk, of zie je een waaiervorm? (→ homoscedasticiteit)

## 4. Multicollineariteit: de VIF
De **Variance Inflation Factor** (VIF) meet per feature hoe sterk hij samenhangt met de andere features.
Vuistregel: een **VIF boven de 5 (soms 10)** wijst op problematische multicollineariteit.
De VIF-functie is al voor je geschreven.

In [ ]:
# VIF-functie (al ingevuld): voor elke feature kijken we hoe goed
# de andere features die feature kunnen voorspellen (R²), en rekenen dat om naar VIF.
def bereken_vif(X_df):
    vifs = {}
    for kolom in X_df.columns:
        andere = [c for c in X_df.columns if c != kolom]
        r2_kolom = LinearRegression().fit(X_df[andere], X_df[kolom]).score(X_df[andere], X_df[kolom])
        vifs[kolom] = 1.0 / (1.0 - r2_kolom) if r2_kolom < 1 else np.inf
    return pd.Series(vifs).sort_values(ascending=False)

### TODO 4 — Bereken de VIF
Roep de functie `bereken_vif` aan op de **features** (gebruik de volledige `X`).

> Tip: `bereken_vif(X)`.

In [ ]:
# ANTWOORD
vif_waarden = bereken_vif(X)
print(vif_waarden.round(1))

> **Interpreteer:** welke features hebben een VIF boven de 5? Wat betekent dat voor het
> vertrouwen in hun coëfficiënten? (Denk terug aan de slide over multicollineariteit.)

## 5. Bonus — Vergelijk met Ridge
Ridge-regressie remt grote coëfficiënten af (regularisatie) en gaat daardoor beter om met
gecorreleerde features. Hier zie je of dat de prestaties verandert.

### TODO 5 (bonus) — Train een Ridge-model
Train een `Ridge`-model met `alpha=1.0`, voorspel de test-set en bereken de R².
Vergelijk met de R² van het gewone lineaire model.

> Tip: `Ridge(alpha=1.0)` werkt verder net als `LinearRegression`.

In [ ]:
# ANTWOORD
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
r2_ridge = r2_score(y_test, y_pred_ridge)

print(f"R² gewoon lineair: {r2:.3f}")
print(f"R² Ridge        : {r2_ridge:.3f}")

## 6. Bespreking
Bespreek met je buur:

- Hoeveel beter is het lineaire model dan de baseline uit opdracht 1?
- Welke twee features hadden het sterkste effect, en is dat logisch?
- Wat zag je in de residuenplot — kloppen de aannames van lineariteit en homoscedasticiteit?
- Welke features hadden een hoge VIF? Wat zou je daaraan kunnen doen?
- Verandert Ridge de prestaties veel? Waarom wel/niet, denk je?

---
### Toelichting voor de trainer
- Verwacht **R² ≈ 0,45** op de test-set: een duidelijke verbetering t.o.v. de baseline (R² ≈ 0).
- Sterkste effecten zijn doorgaans **s1** (negatief) en **s5** / **bmi** (positief).
- De residuenplot oogt redelijk willekeurig; bespreek dat lichte patronen normaal zijn.
- **VIF**: `s1` en `s2` zijn sterk gecorreleerd (VIF ≈ 59 en 39) — een mooi praktijkvoorbeeld van
  multicollineariteit. Mogelijke oplossing: één van beide verwijderen, of regularisatie (Ridge).
- Ridge verandert de R² hier nauwelijks, maar maakt de coëfficiënten stabieler — goed gesprekspunt.